<a href="https://colab.research.google.com/github/JVV06/Synth-dataset/blob/main/TGAN_preprocessing_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

# Upload file in Colab first
from google.colab import files
uploaded = files.upload()


Saving health_indicators_ready.csv to health_indicators_ready (1).csv


In [ ]:
# Load dataset
df = pd.read_csv(list(uploaded.keys())[0], sep=';')

# Preview
df.head()

,GHO (DISPLAY),YEAR (DISPLAY),DIMENSION (NAME),Numeric
0,Population using at least basic sanitation ser...,2017.0,Urban,75.223231943
1,Prevalence of overweight among children and ad...,2017.0,Female,28.953755
2,Tobacco Warning: W2c smokeless tobacco packagi...,2014.0,The quit line number must appear on smokeless ...,NaN
3,Proportion of population with primary reliance...,1995.0,Rural,4.4
4,"Mean Non-HDL cholesterol, crude",1993.0,Male,3.0


In [ ]:
# Standardize column names (important)
df.columns = df.columns.str.strip().str.replace(" ", "_")

# Example expected names:
# Indicator, Year, Dimension, Numeric

# Drop missing values
df = df.dropna()

# Remove duplicates
df = df.drop_duplicates()

print("Shape after cleaning:", df.shape)

Shape after cleaning: (15338, 4)


**Rename Columns (for clarity)**

In [ ]:
df = df.rename(columns={
    'GHO_(DISPLAY)': 'Indicator',
    'YEAR_(DISPLAY)': 'Year',
    'DIMENSION_(NAME)': 'Dimension',
    'Numeric': 'Value'
})

In [ ]:
# Preview
df.head()

,Indicator,Year,Dimension,Value
0,Population using at least basic sanitation ser...,2017.0,Urban,75.223231943
1,Prevalence of overweight among children and ad...,2017.0,Female,28.953755
3,Proportion of population with primary reliance...,1995.0,Rural,4.4
4,"Mean Non-HDL cholesterol, crude",1993.0,Male,3.0
8,Proportion of population with primary reliance...,2007.0,Urban,82.4


**Ensure Correct Data Types**

In [ ]:
# Convert Year to integer
df['Year'] = df['Year'].astype(int)

# Convert Value to numeric
df['Value'] = pd.to_numeric(df['Value'], errors='coerce')

# Drop rows where Value failed conversion
df = df.dropna(subset=['Value'])

# Reset index
df = df.reset_index(drop=True)

**Reduce Cardinality (IMPORTANT for GAN performance)**

In [ ]:
# Keep top 20 most frequent indicators
top_indicators = df['Indicator'].value_counts().nlargest(20).index

df = df[df['Indicator'].isin(top_indicators)]

print("Remaining indicators:", df['Indicator'].nunique())

Remaining indicators: 20


**Encode Categorical Variables (CTGAN handles categories, but optional encoding helps)**

For CTGAN (SDV), you actually keep them as strings, NOT numeric encoding.

So DO NOT label encode — instead:

In [ ]:
# Ensure categorical columns are strings
df['Indicator'] = df['Indicator'].astype(str)
df['Dimension'] = df['Dimension'].astype(str)

**Normalize Numerical Variable (Optional but recommended)**

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

df['Value_scaled'] = scaler.fit_transform(df[['Value']])

You can either:

Use Value directly (CTGAN handles it)
OR use Value_scaled (more stable training)

**Final Dataset for GAN**

Select only required columns:

In [ ]:
df_final = df[['Indicator', 'Year', 'Dimension', 'Value']]

df_final.head()

,Indicator,Year,Dimension,Value
1,Prevalence of overweight among children and ad...,2017,Female,28.953755
2,Proportion of population with primary reliance...,1995,Rural,4.400000
4,Proportion of population with primary reliance...,2007,Urban,82.400000
6,Prevalence of thinness among children and adol...,2005,Male,6.880980
8,"Alcohol, recorded per capita (15+) consumption...",2018,Other alcoholic beverages,0.730000


**Save Clean Dataset (important for reproducibility)**

In [ ]:
df_final.to_csv("clean_health_data.csv", index=False)

files.download("clean_health_data.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Define Metadata for CTGAN (VERY IMPORTANT STEP)**

In [ ]:
# Install sdv library if not already installed
!pip install sdv

from sdv.metadata import SingleTableMetadata

metadata = SingleTableMetadata()

metadata.detect_from_dataframe(df_final)

metadata

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.6/202.6 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.7/14.7 MB 64.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.3/202.3 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 75.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 7.0 MB/s eta 0:00:00


{
    "METADATA_SPEC_VERSION": "SINGLE_TABLE_V1",
    "columns": {
        "Indicator": {
            "sdtype": "categorical"
        },
        "Year": {
            "sdtype": "numerical"
        },
        "Dimension": {
            "sdtype": "categorical"
        },
        "Value": {
            "sdtype": "numerical"
        }
    }
}

**Identify Column Types Explicitly**

In [ ]:
metadata.update_column(
    column_name='Indicator',
    sdtype='categorical'
)

metadata.update_column(
    column_name='Dimension',
    sdtype='categorical'
)

metadata.update_column(
    column_name='Year',
    sdtype='numerical'
)

metadata.update_column(
    column_name='Value',
    sdtype='numerical'
)

**Final Check Before GAN**

In [ ]:
print(df_final.info())
print(df_final.head())

<class 'pandas.core.frame.DataFrame'>
Index: 7351 entries, 1 to 15325
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Indicator  7351 non-null   object 
 1   Year       7351 non-null   int64  
 2   Dimension  7351 non-null   object 
 3   Value      7351 non-null   float64
dtypes: float64(1), int64(1), object(2)
memory usage: 287.1+ KB
None
                                           Indicator  Year  \
1  Prevalence of overweight among children and ad...  2017   
2  Proportion of population with primary reliance...  1995   
4  Proportion of population with primary reliance...  2007   
6  Prevalence of thinness among children and adol...  2005   
8  Alcohol, recorded per capita (15+) consumption...  2018   

                   Dimension      Value  
1                     Female  28.953755  
2                      Rural   4.400000  
4                      Urban  82.400000  
6                       Male   6.880980  
8  Oth

**Key Notes (Important for the Paper)**



The dataset was preprocessed by removing missing values, standardizing variable formats, and selecting key features including health indicators, temporal attributes, demographic dimensions, and numerical values. Categorical variables were preserved in their original format to maintain semantic integrity for generative modelling, while numerical variables were optionally normalized to improve model stability. Additionally, feature reduction techniques were applied to limit high-cardinality categorical variables, thereby enhancing GAN training efficiency.

In [ ]:
!pip install sdv